## 15. Inventory Risk Framework

The Inventory Risk Framework translates forecasting output into actionable inventory risk segments.

The objective is to identify product-store combinations that may require operational attention based on forecasted demand, stockout exposure, and product importance.

This framework helps connect demand forecasting with inventory decision-making by identifying critical products, high-risk availability cases, constrained demand candidates, and lower-priority items.

### 15.1 Load Forecast Output

The evaluation forecast output is loaded as the main input for the inventory risk framework.

This file contains actual sales, final forecasted sales, stockout information, product-store identifiers, and business context variables.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_folder = Path(r"C:\Users\safwatr\IntelGraphicsProfiles\Fresh Retail")
processed_folder = project_folder / "processed"

forecast_output = pd.read_csv(
    processed_folder / "eval_forecast_output.csv"
)

forecast_output["dt"] = pd.to_datetime(forecast_output["dt"])

print("Forecast output shape:", forecast_output.shape)
print("Date range:", forecast_output["dt"].min(), "to", forecast_output["dt"].max())

display(forecast_output.head())

Forecast output shape: (350000, 47)
Date range: 2024-06-26 00:00:00 to 2024-07-02 00:00:00


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,stock_hour6_22_cnt,...,weekend_holiday_interaction,pred_hgb,pred_lgb_base,pred_lgb_tuned,ensemble_prediction,stockout_severity,stockout_correction_factor,final_prediction,absolute_error,forecast_error
0,0,0,2,29,78,82,4,2024-06-26,0.0,0,...,0,0.423789,0.413327,0.353310,0.385411,No Stockout,1.0,0.385411,0.385411,-0.385411
1,0,0,2,29,78,82,4,2024-06-27,0.2,0,...,0,0.276713,-0.001713,0.086517,0.098087,No Stockout,1.0,0.098087,0.101913,0.101913
2,0,0,2,29,78,82,4,2024-06-28,0.8,6,...,0,0.577090,0.450286,0.449803,0.475405,Medium Stockout,1.0,0.475405,0.324595,0.324595
3,0,0,2,29,78,82,4,2024-06-29,0.0,16,...,1,0.646364,0.757211,0.563898,0.638385,Full-Day Stockout,0.0,0.000000,0.000000,0.000000
4,0,0,2,29,78,82,4,2024-06-30,1.6,0,...,1,0.530162,0.659033,0.422997,0.515241,No Stockout,1.0,0.515241,1.084759,1.084759


### 15.2 Add ABC Product Classes

ABC product classes are added to the forecast output to include product importance in the inventory risk framework.

ABC classification was created earlier based on each product's contribution to total sales:

- A products: highest sales contribution
- B products: medium sales contribution
- C products: lower sales contribution

Adding ABC classes helps prioritize inventory risk. A high-risk A product is more important operationally than a high-risk C product because A products contribute the majority of sales.

In [2]:
product_abc = pd.read_csv(
    processed_folder / "eda_product_sales_contribution_abc.csv"
)

product_abc = product_abc[["product_id", "abc_class"]].drop_duplicates()

print("ABC product class shape:", product_abc.shape)
display(product_abc.head())

forecast_risk_base = forecast_output.merge(
    product_abc,
    on="product_id",
    how="left"
)

print("Forecast risk base shape:", forecast_risk_base.shape)
print("Missing ABC classes:", forecast_risk_base["abc_class"].isna().sum())

display(forecast_risk_base.head())

ABC product class shape: (865, 2)


,product_id,abc_class
0,267,A
1,4,A
2,300,A
3,117,A
4,215,A


Forecast risk base shape: (350000, 48)
Missing ABC classes: 0


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,stock_hour6_22_cnt,...,pred_hgb,pred_lgb_base,pred_lgb_tuned,ensemble_prediction,stockout_severity,stockout_correction_factor,final_prediction,absolute_error,forecast_error,abc_class
0,0,0,2,29,78,82,4,2024-06-26,0.0,0,...,0.423789,0.413327,0.353310,0.385411,No Stockout,1.0,0.385411,0.385411,-0.385411,A
1,0,0,2,29,78,82,4,2024-06-27,0.2,0,...,0.276713,-0.001713,0.086517,0.098087,No Stockout,1.0,0.098087,0.101913,0.101913,A
2,0,0,2,29,78,82,4,2024-06-28,0.8,6,...,0.577090,0.450286,0.449803,0.475405,Medium Stockout,1.0,0.475405,0.324595,0.324595,A
3,0,0,2,29,78,82,4,2024-06-29,0.0,16,...,0.646364,0.757211,0.563898,0.638385,Full-Day Stockout,0.0,0.000000,0.000000,0.000000,A
4,0,0,2,29,78,82,4,2024-06-30,1.6,0,...,0.530162,0.659033,0.422997,0.515241,No Stockout,1.0,0.515241,1.084759,1.084759,A


### 15.3 Create Forecast Demand Tiers

Forecast demand tiers are created from the final predicted sales values.

The objective is to classify product-store-date observations into demand levels such as low, medium, high, and very high forecast demand.

Using demand tiers makes the inventory risk framework easier to interpret and supports business prioritization.

In [3]:
forecast_quantiles = forecast_risk_base["final_prediction"].quantile(
    [0.50, 0.75, 0.90]
)

low_threshold = forecast_quantiles.loc[0.50]
high_threshold = forecast_quantiles.loc[0.75]
very_high_threshold = forecast_quantiles.loc[0.90]

print("Forecast demand thresholds:")
print("Low/Medium threshold:", low_threshold)
print("Medium/High threshold:", high_threshold)
print("High/Very High threshold:", very_high_threshold)

def classify_forecast_demand(value):
    if value <= low_threshold:
        return "Low Forecast Demand"
    elif value <= high_threshold:
        return "Medium Forecast Demand"
    elif value <= very_high_threshold:
        return "High Forecast Demand"
    else:
        return "Very High Forecast Demand"

forecast_risk_base["forecast_demand_tier"] = (
    forecast_risk_base["final_prediction"]
    .apply(classify_forecast_demand)
)

forecast_demand_tier_summary = (
    forecast_risk_base
    .groupby("forecast_demand_tier")
    .agg(
        observations=("final_prediction", "count"),
        total_forecast_sales=("final_prediction", "sum"),
        avg_forecast_sales=("final_prediction", "mean"),
        total_actual_sales=("sale_amount", "sum"),
        avg_actual_sales=("sale_amount", "mean")
    )
    .reset_index()
)

display(forecast_demand_tier_summary)

Forecast demand thresholds:
Low/Medium threshold: 0.6623651259138413
Medium/High threshold: 1.114224483437207
High/Very High threshold: 2.023597688945319


,forecast_demand_tier,observations,total_forecast_sales,avg_forecast_sales,total_actual_sales,avg_actual_sales
0,High Forecast Demand,52500,77065.971256,1.467923,85337.052,1.625468
1,Low Forecast Demand,175000,75623.318633,0.432133,89637.727,0.512216
2,Medium Forecast Demand,87500,74073.316809,0.846552,83064.809,0.949312
3,Very High Forecast Demand,35000,141072.330786,4.030638,159528.935,4.557970


### 15.4 Create Stockout Risk Tiers

Stockout risk tiers are created from stockout severity levels.

The objective is to translate operational stockout exposure into clear availability risk levels that can be used in the inventory risk framework.

The stockout risk tiers are defined as:

- No Stockout → No Availability Risk
- Low Stockout → Low Availability Risk
- Medium Stockout → Medium Availability Risk
- High Stockout → High Availability Risk
- Full-Day Stockout → Critical Availability Risk

These tiers help identify product-store-date observations where product availability may limit realized sales.

In [4]:
def classify_stockout_risk_tier(stockout_severity):
    if stockout_severity == "No Stockout":
        return "No Availability Risk"
    elif stockout_severity == "Low Stockout":
        return "Low Availability Risk"
    elif stockout_severity == "Medium Stockout":
        return "Medium Availability Risk"
    elif stockout_severity == "High Stockout":
        return "High Availability Risk"
    elif stockout_severity == "Full-Day Stockout":
        return "Critical Availability Risk"
    else:
        return "Unknown Availability Risk"

forecast_risk_base["stockout_risk_tier"] = (
    forecast_risk_base["stockout_severity"]
    .apply(classify_stockout_risk_tier)
)

stockout_risk_tier_summary = (
    forecast_risk_base
    .groupby("stockout_risk_tier")
    .agg(
        observations=("stockout_risk_tier", "count"),
        total_actual_sales=("sale_amount", "sum"),
        avg_actual_sales=("sale_amount", "mean"),
        total_forecast_sales=("final_prediction", "sum"),
        avg_forecast_sales=("final_prediction", "mean"),
        total_stockout_hours=("stock_hour6_22_cnt", "sum"),
        avg_stockout_hours=("stock_hour6_22_cnt", "mean")
    )
    .reset_index()
)

display(stockout_risk_tier_summary)

,stockout_risk_tier,observations,total_actual_sales,avg_actual_sales,total_forecast_sales,avg_forecast_sales,total_stockout_hours,avg_stockout_hours
0,Critical Availability Risk,12541,427.660,0.034101,0.000000,0.000000,200656,16.000000
1,High Availability Risk,35361,29480.520,0.833702,27622.562789,0.781159,389713,11.020984
2,Low Availability Risk,46283,70273.843,1.518351,56880.420368,1.228970,120414,2.601690
3,Medium Availability Risk,49253,61845.270,1.255665,53861.092094,1.093560,303248,6.156945
4,No Availability Risk,206562,255541.230,1.237116,229470.862233,1.110906,0,0.000000


### 15.5 Create Inventory Risk Segments

Inventory risk segments combine product importance, forecasted demand, and stockout risk.

The goal is to identify product-store-date observations that require different levels of operational attention.

The segmentation logic is designed to prioritize:

- Important products with high forecasted demand and high availability risk
- Cases where low observed sales may be caused by stockout constraints
- Lower-priority products with low forecasted demand and low availability risk

In [5]:
def classify_inventory_risk(row):
    abc_class = row["abc_class"]
    demand_tier = row["forecast_demand_tier"]
    stockout_risk = row["stockout_risk_tier"]
    actual_sales = row["sale_amount"]
    forecast_sales = row["final_prediction"]
    
    # Highest risk: important product + strong forecast + high/critical stockout risk
    if (
        abc_class == "A"
        and demand_tier in ["High Forecast Demand", "Very High Forecast Demand"]
        and stockout_risk in ["High Availability Risk", "Critical Availability Risk"]
    ):
        return "Critical Risk"
    
    # High forecast + high availability risk, even if not A-class
    elif (
        demand_tier in ["High Forecast Demand", "Very High Forecast Demand"]
        and stockout_risk in ["High Availability Risk", "Critical Availability Risk"]
    ):
        return "High Priority"
    
    # Potential constrained demand: low realized sales but stockout risk is high
    elif (
        actual_sales <= 0.2
        and forecast_sales > 0.5
        and stockout_risk in ["High Availability Risk", "Critical Availability Risk"]
    ):
        return "Constrained Demand Candidate"
    
    # Important demand but no severe stockout
    elif (
        abc_class == "A"
        and demand_tier in ["High Forecast Demand", "Very High Forecast Demand"]
        and stockout_risk in ["No Availability Risk", "Low Availability Risk", "Medium Availability Risk"]
    ):
        return "High Demand Monitor"
    
    # Medium-level monitoring
    elif (
        demand_tier in ["Medium Forecast Demand", "High Forecast Demand"]
        or stockout_risk in ["Medium Availability Risk", "High Availability Risk"]
    ):
        return "Monitor"
    
    # Low demand and low stockout exposure
    else:
        return "Low Priority"

forecast_risk_base["inventory_risk_segment"] = forecast_risk_base.apply(
    classify_inventory_risk,
    axis=1
)

inventory_risk_summary = (
    forecast_risk_base
    .groupby("inventory_risk_segment")
    .agg(
        observations=("inventory_risk_segment", "count"),
        unique_products=("product_id", "nunique"),
        unique_stores=("store_id", "nunique"),
        total_actual_sales=("sale_amount", "sum"),
        total_forecast_sales=("final_prediction", "sum"),
        avg_actual_sales=("sale_amount", "mean"),
        avg_forecast_sales=("final_prediction", "mean"),
        total_stockout_hours=("stock_hour6_22_cnt", "sum"),
        avg_stockout_hours=("stock_hour6_22_cnt", "mean")
    )
    .reset_index()
    .sort_values("total_forecast_sales", ascending=False)
)

display(inventory_risk_summary)

,inventory_risk_segment,observations,unique_products,unique_stores,total_actual_sales,total_forecast_sales,avg_actual_sales,avg_forecast_sales,total_stockout_hours,avg_stockout_hours
2,High Demand Monitor,74175,117,885,219390.603,193791.184535,2.957743,2.612621,113953,1.536272
5,Monitor,135762,816,898,118241.384,101180.552571,0.870946,0.745279,563323,4.149342
4,Low Priority,132531,844,898,67914.846,59403.675233,0.512445,0.448225,251815,1.900046
1,Critical Risk,5936,93,746,11076.800,11953.436660,1.866038,2.013719,65216,10.986523
0,Constrained Demand Candidate,1171,181,606,183.690,813.193286,0.156866,0.694443,15017,12.824082
3,High Priority,425,73,225,761.200,692.895198,1.791059,1.630342,4707,11.075294


### 15.6 Add Recommended Actions

Recommended actions are added to each inventory risk segment.

The objective is to translate analytical risk segments into clear operational guidance for replenishment, monitoring, and business review.

Each segment receives a recommended action based on its demand level, stockout exposure, and business importance.

In [6]:
def assign_recommended_action(segment):
    if segment == "Critical Risk":
        return "Immediate replenishment review and availability monitoring"
    elif segment == "High Priority":
        return "Prioritize stock review and monitor near-term demand"
    elif segment == "Constrained Demand Candidate":
        return "Investigate stock availability and potential suppressed demand"
    elif segment == "High Demand Monitor":
        return "Maintain strong availability and monitor demand trend"
    elif segment == "Monitor":
        return "Monitor regularly and review if stockout increases"
    elif segment == "Low Priority":
        return "Standard replenishment process"
    else:
        return "Review manually"

forecast_risk_base["recommended_action"] = (
    forecast_risk_base["inventory_risk_segment"]
    .apply(assign_recommended_action)
)

recommended_action_summary = (
    forecast_risk_base
    .groupby(["inventory_risk_segment", "recommended_action"])
    .agg(
        observations=("inventory_risk_segment", "count"),
        unique_products=("product_id", "nunique"),
        unique_stores=("store_id", "nunique"),
        total_actual_sales=("sale_amount", "sum"),
        total_forecast_sales=("final_prediction", "sum"),
        total_stockout_hours=("stock_hour6_22_cnt", "sum"),
        avg_stockout_hours=("stock_hour6_22_cnt", "mean")
    )
    .reset_index()
    .sort_values("total_forecast_sales", ascending=False)
)

display(recommended_action_summary)

,inventory_risk_segment,recommended_action,observations,unique_products,unique_stores,total_actual_sales,total_forecast_sales,total_stockout_hours,avg_stockout_hours
2,High Demand Monitor,Maintain strong availability and monitor deman...,74175,117,885,219390.603,193791.184535,113953,1.536272
5,Monitor,Monitor regularly and review if stockout incre...,135762,816,898,118241.384,101180.552571,563323,4.149342
4,Low Priority,Standard replenishment process,132531,844,898,67914.846,59403.675233,251815,1.900046
1,Critical Risk,Immediate replenishment review and availabilit...,5936,93,746,11076.800,11953.436660,65216,10.986523
0,Constrained Demand Candidate,Investigate stock availability and potential s...,1171,181,606,183.690,813.193286,15017,12.824082
3,High Priority,Prioritize stock review and monitor near-term ...,425,73,225,761.200,692.895198,4707,11.075294


#### 15.6 Recommended Actions — Result

Recommended actions were assigned to each inventory risk segment to translate analytical results into operational guidance.

The framework identified six main inventory risk segments:

- **Critical Risk**
- **High Priority**
- **Constrained Demand Candidate**
- **High Demand Monitor**
- **Monitor**
- **Low Priority**

The **High Demand Monitor** segment contained the largest sales volume, with approximately **219.4K actual sales** and **193.8K forecasted sales**. This segment represents high-demand observations that currently have relatively manageable stockout exposure. The recommended action is to maintain strong product availability and monitor demand trends.

The **Critical Risk** segment contained **5,936 observations** across **93 products** and **746 stores**, with average stockout hours of approximately **10.99**. These observations require immediate replenishment review and availability monitoring because they combine meaningful demand with high stockout exposure.

The **Constrained Demand Candidate** segment contained observations with very low actual sales but high stockout exposure and higher forecasted demand. This segment is important because it may represent suppressed demand caused by product unavailability rather than weak customer demand.

The **Monitor** and **Low Priority** segments represent broader operational groups that should be managed through regular monitoring or standard replenishment processes.

Overall, the recommended action layer makes the Inventory Risk Framework actionable by linking each risk segment to a clear business response.

15.6 Recommended Actions

Recommended actions were assigned to each inventory risk segment to convert the analytical framework into practical operational guidance. The goal was to make the inventory risk output usable for replenishment review, business monitoring, and dashboard reporting.

The framework identified six inventory risk segments: Critical Risk, High Priority, Constrained Demand Candidate, High Demand Monitor, Monitor, and Low Priority.

The Critical Risk segment represents the highest-priority cases. These observations combine strong demand signals with high stockout exposure. This segment contained 5,936 observations across 93 products and 746 stores, with average stockout hours of approximately 10.99. The recommended action for this group is immediate replenishment review and availability monitoring.

The High Demand Monitor segment contained the largest sales volume, with approximately 219.4K actual sales and 193.8K forecasted sales. These observations represent high-demand product-store-date cases where stockout exposure is not yet critical. The recommended action is to maintain strong availability and continue monitoring the demand trend.

The Constrained Demand Candidate segment is especially important from an inventory intelligence perspective. This group had very low actual sales but higher forecasted sales and very high average stockout exposure. This suggests that some low observed sales may reflect product unavailability rather than weak demand. The recommended action is to investigate stock availability and potential suppressed demand.

The Monitor segment represents observations that require regular review, especially if stockout exposure increases. The Low Priority segment includes lower-demand or lower-risk cases and can be managed through the standard replenishment process.

Overall, the recommended action layer strengthens the practical value of the Inventory Risk Framework by turning forecast and stockout analysis into clear operational priorities.

In [7]:
forecast_risk_base.to_csv(
    processed_folder / "inventory_risk_output.csv",
    index=False,
    encoding="utf-8-sig"
)

inventory_risk_summary.to_csv(
    processed_folder / "inventory_risk_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

recommended_action_summary.to_csv(
    processed_folder / "inventory_recommended_action_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

forecast_demand_tier_summary.to_csv(
    processed_folder / "inventory_forecast_demand_tier_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

stockout_risk_tier_summary.to_csv(
    processed_folder / "inventory_stockout_risk_tier_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Inventory risk framework outputs saved successfully.")

Inventory risk framework outputs saved successfully.


#### 15. Inventory Risk Framework — Result

The Inventory Risk Framework was successfully created using the evaluation forecast output.

The framework combines:

- Forecasted demand
- Stockout severity
- ABC product class
- Product-store identifiers
- Actual sales
- Forecasted sales

Forecast demand tiers were created using prediction percentiles. These tiers classified observations into Low, Medium, High, and Very High Forecast Demand.

Stockout risk tiers were created from stockout severity and mapped to availability risk levels ranging from No Availability Risk to Critical Availability Risk.

The final inventory risk segmentation produced six actionable segments:

- Critical Risk
- High Priority
- Constrained Demand Candidate
- High Demand Monitor
- Monitor
- Low Priority

The High Demand Monitor segment contained the largest sales volume, with approximately 219.4K actual sales and 193.8K forecasted sales. These cases require strong availability monitoring to prevent future stockout risk.

The Critical Risk segment contained 5,936 observations across 93 products and 746 stores, with average stockout hours of approximately 10.99. These cases require immediate replenishment review and availability monitoring.

The Constrained Demand Candidate segment identified cases where actual sales were very low, but forecasted demand was higher and stockout exposure was severe. These observations may represent suppressed demand caused by product unavailability.

Recommended actions were assigned to each risk segment to translate the analytical framework into operational guidance.

The inventory risk outputs were saved successfully for reporting, Power BI dashboard development, and business recommendation generation.